# EEGSynthesizer — validación doctoral bloqueada

Este notebook separa estrictamente desarrollo y evaluación externa. Los casos
`chb01`, `chb02`, `chb03`, `chb05`, `chb06` y el registro `chb21` —la misma
persona que `chb01`— se reservan para desarrollo. Los demás sujetos permanecen
bloqueados hasta que el generador tenga manifiesto congelado.

Los nombres `generalized_absence` y `focal_temporal` son escenarios paramétricos,
no diagnósticos. CHB-MIT se usa para contraste ictal/interictal, fidelidad y
transferencia; no certifica subtipos clínicos. La proximidad se calcula solo en
el espacio de características y no constituye privacidad formal.

In [ ]:
# BLOQUE 0 — configuración y modos de ejecución
import gc, hashlib, json, os, re, shutil, subprocess, sys, time, urllib.error, urllib.request
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd
import mne
from scipy import signal, stats
from scipy.spatial.distance import cdist
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
SYN_DIR = PROJECT_ROOT / "dataset_eeg_final"
REAL_DIR = PROJECT_ROOT / "dataset_doctorado_final"
OUT_DIR = REAL_DIR / "validation_q1_assets"
CACHE = REAL_DIR / "chb_mit_cache"
WORK_ROOT = PROJECT_ROOT / "tmp" / "eegsynth_rebuild"
DEV_WORK = WORK_ROOT / "dev_candidate"
REAL_WORK = WORK_ROOT / "real_candidate"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

MODE = os.environ.get("EEGSYN_VALIDATION_MODE", "status").strip().lower()
if MODE not in {"status", "prepare_dev", "external"}:
    raise ValueError("EEGSYN_VALIDATION_MODE debe ser status, prepare_dev o external")
REBUILD_REAL = os.environ.get("EEGSYN_REBUILD_REAL", "0") == "1"
DOWNLOAD_WORKERS = max(1, int(os.environ.get("EEGSYN_DOWNLOAD_WORKERS", "4")))
FS = 250
WIN_SEC = 2.0
WIN_PTS = int(FS * WIN_SEC)
SEED = 42
MAX_WINDOWS_PER_SUBJECT_CLASS = 400
DEV_CASES = {"chb01", "chb02", "chb03", "chb05", "chb06"}
CASE_PERSON = {"chb21": "chb01"}

REF_CH = ["Fp1","Fp2","F7","F3","Fz","F4","F8","T3","C3","Cz","C4","T4","T5","P3","Pz","P4","T6","O1","O2"]
BIPOLAR = [
    ("FP1-F7","FP1","F7"),("F7-T3","F7","T3"),("T3-T5","T3","T5"),("T5-O1","T5","O1"),
    ("FP1-F3","FP1","F3"),("F3-C3","F3","C3"),("C3-P3","C3","P3"),("P3-O1","P3","O1"),
    ("FP2-F4","FP2","F4"),("F4-C4","F4","C4"),("C4-P4","C4","P4"),("P4-O2","P4","O2"),
    ("FP2-F8","FP2","F8"),("F8-T4","F8","T4"),("T4-T6","T4","T6"),("T6-O2","T6","O2"),
    ("FZ-CZ","FZ","CZ"),("CZ-PZ","CZ","PZ"),
]
FEATURE_NAMES = ["ptp_med","ptp_p95","std_med","std_p95","bp1_4","bp4_8","bp8_13","bp13_30",
                 "ratio_2_6__6_20","beta","Hspec","corr_abs_mean","corr_abs_p95","dom_freq",
                 "freq_first","freq_last","rms_last_first","spatial_concentration","laterality_abs"]
FEATURE_DOMAINS = {
    "background":[0,1,4,5,6,7,9,10], "temporal":[8,13,14,15,16],
    "spatial":[11,12,17,18], "variability":list(range(len(FEATURE_NAMES))),
}
print("MODE:", MODE, "REBUILD_REAL:", REBUILD_REAL, "DOWNLOAD_WORKERS:", DOWNLOAD_WORKERS)

In [ ]:
# BLOQUE 1 — integridad, montaje común y características
def sha256_file(path, block=16*1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(block),b""): h.update(b)
    return h.hexdigest()

def stable_int(value):
    return int.from_bytes(hashlib.sha256(str(value).encode()).digest()[:4],"little")

def load_synthetic(require_frozen=False):
    mp=SYN_DIR/"generation_manifest.json"
    required=["X_train.npy","X_val.npy","X_test.npy","y_train.npy","y_val.npy","y_test.npy",
              "cohort_metadata.csv","window_metadata_train.csv","window_metadata_val.csv","window_metadata_test.csv"]
    if not mp.exists(): return {"ready":False,"reason":"falta generation_manifest.json"}
    manifest=json.loads(mp.read_text(encoding="utf-8"))
    if int(str(manifest.get("config",{}).get("schema_version","0")).split(".")[0])<4:
        return {"ready":False,"reason":"schema sintético anterior a 4"}
    missing=[n for n in required if not (SYN_DIR/n).exists()]
    if missing: return {"ready":False,"reason":f"faltan artefactos: {missing}"}
    bad=[]
    for rel,info in manifest.get("files",{}).items():
        p=SYN_DIR/rel
        if not p.exists() or sha256_file(p)!=info.get("sha256"): bad.append(rel)
    if bad: return {"ready":False,"reason":f"SHA256 inválido: {bad[:8]}"}
    if require_frozen and not manifest.get("external_freeze",{}).get("frozen",False):
        return {"ready":False,"reason":"el generador no fue congelado antes de evaluación externa"}
    return {"ready":True,"manifest":manifest,
            "X":{k:np.load(SYN_DIR/f"X_{k}.npy",mmap_mode="r") for k in ("train","val","test")},
            "y":{k:np.load(SYN_DIR/f"y_{k}.npy") for k in ("train","val","test")},
            "meta":{k:pd.read_csv(SYN_DIR/f"window_metadata_{k}.csv") for k in ("train","val","test")},
            "cohort":pd.read_csv(SYN_DIR/"cohort_metadata.csv")}

def synthetic_to_bipolar(X):
    idx={c.upper():i for i,c in enumerate(REF_CH)}
    return np.stack([np.asarray(X)[:,:,idx[a]]-np.asarray(X)[:,:,idx[b]] for _,a,b in BIPOLAR],axis=2).astype(np.float32)

def norm_name(x):
    x=x.upper().replace("EEG","").replace("-REF","").replace(" ","")
    x=re.sub(r"-(?:0|1)$","",x)
    return x.replace("T7","T3").replace("P7","T5").replace("T8","T4").replace("P8","T6")

def common_bipolar(data_ct,names):
    lookup={}
    for i,name in enumerate(names):
        q=norm_name(name)
        if "-" in q:
            a,b=q.split("-",1)
            if (a,b) not in lookup:
                lookup[(a,b)]=(i,1.0); lookup[(b,a)]=(i,-1.0)
    out=[]
    for _,a,b in BIPOLAR:
        if (a,b) not in lookup: return None
        i,sgn=lookup[(a,b)]; out.append(sgn*data_ct[i])
    return np.asarray(out,dtype=np.float32).T

def one_feature(window,zscore=False):
    x=np.asarray(window,dtype=np.float64)
    if zscore: x=(x-x.mean(0,keepdims=True))/(x.std(0,keepdims=True)+1e-8)
    ptp=np.ptp(x,axis=0); sd=x.std(axis=0)
    f,pch=signal.welch(x,fs=FS,nperseg=min(256,len(x)),axis=0); p=pch.mean(axis=1)
    def bp(a,b):
        m=(f>=a)&(f<b); return float(np.trapezoid(p[m],f[m])) if m.sum()>=2 else 0.0
    m=(f>=1)&(f<=30)&(p>0); beta=float(np.polyfit(np.log(f[m]),np.log(p[m]),1)[0]) if m.sum()>=3 else 0.0
    pn=p/(p.sum()+1e-12); H=float(-(pn[pn>0]*np.log(pn[pn>0])).sum()/np.log(max(2,len(pn))))
    C=np.corrcoef(x,rowvar=False); cv=np.abs(C[np.triu_indices_from(C,k=1)]); cv=cv[np.isfinite(cv)]
    band=(f>=1)&(f<=20); dom=float(f[band][np.argmax(p[band])]) if band.any() else 0.0
    third=max(FS//2,len(x)//3)
    def dompart(q):
        fq,pq=signal.welch(q,fs=FS,nperseg=min(256,len(q)),axis=0); pq=pq.mean(1); mm=(fq>=1)&(fq<=20)
        return float(fq[mm][np.argmax(pq[mm])]) if mm.any() else 0.0
    ff,fl=dompart(x[:third]),dompart(x[-third:])
    rf=float(np.sqrt(np.mean(x[-third:]**2))/(np.sqrt(np.mean(x[:third]**2))+1e-12))
    concentration=float(ptp.max()/(ptp.sum()+1e-12))
    left=float(ptp[:8].sum()); right=float(ptp[8:16].sum()); lateral=abs(left-right)/(left+right+1e-12)
    return [float(np.median(ptp)),float(np.percentile(ptp,95)),float(np.median(sd)),float(np.percentile(sd,95)),
            bp(1,4),bp(4,8),bp(8,13),bp(13,30),bp(2,6)/(bp(6,20)+1e-12),beta,H,
            float(cv.mean()) if len(cv) else 0.0,float(np.percentile(cv,95)) if len(cv) else 0.0,
            dom,ff,fl,rf,concentration,float(lateral)]

def extract_features(X,zscore=False,tag=""):
    F=np.asarray([one_feature(w,zscore=zscore) for w in X],dtype=np.float64)
    if not np.isfinite(F).all(): raise ValueError(f"NaN/Inf en características {tag}")
    print("features",tag,F.shape); return F

SYN=load_synthetic(require_frozen=(MODE=="external"))
print("SYN:","SUPPORTED" if SYN["ready"] else "NOT EVALUABLE — "+SYN["reason"])

In [ ]:
# BLOQUE 2 — selección oficial y descarga reanudable de CHB-MIT
BASE_URL="https://physionet.org/files/chbmit/1.0.0/"
DATA_URL="https://physionet-open.s3.amazonaws.com/chbmit/1.0.0/"

def fetch_text(name,retries=5):
    for attempt in range(retries):
        try: return urllib.request.urlopen(BASE_URL+name,timeout=90).read().decode("utf-8",errors="replace")
        except Exception:
            if attempt+1==retries: raise
            time.sleep(min(30,2**attempt))

def parse_summary(text):
    events={}; current=None; start=None
    for line in text.splitlines():
        mf=re.search(r"File Name:\s*(\S+)",line,re.I)
        if mf: current=mf.group(1); start=None
        ms=re.search(r"Seizure(?: \d+)? Start Time:\s*([0-9.]+)",line,re.I)
        me=re.search(r"Seizure(?: \d+)? End Time:\s*([0-9.]+)",line,re.I)
        if ms: start=float(ms.group(1))
        if me and current and start is not None:
            events.setdefault(current,[]).append((start,float(me.group(1))))
    return events

def official_index(scope):
    records=[x.strip() for x in fetch_text("RECORDS").splitlines() if x.strip().endswith(".edf")]
    seizure_records=set(x.strip() for x in fetch_text("RECORDS-WITH-SEIZURES").splitlines() if x.strip())
    checksums={}
    for line in fetch_text("SHA256SUMS.txt").splitlines():
        m=re.match(r"([0-9a-fA-F]{64})\s+\*?(.+)",line.strip())
        if m: checksums[m.group(2).lstrip("./")]=m.group(1).lower()
    by={}
    for rel in records: by.setdefault(rel.split("/")[0],[]).append(rel)
    selected=[]; events={}
    for case,rels in sorted(by.items()):
        person=CASE_PERSON.get(case,case); is_dev=person in DEV_CASES
        if scope=="dev" and not is_dev: continue
        ictal=sorted(r for r in rels if r in seizure_records)
        if not ictal: continue
        controls=sorted(r for r in rels if r not in seizure_records)[:(2 if is_dev else 1)]
        selected.extend(ictal+controls)
        summary=parse_summary(fetch_text(f"{case}/{case}-summary.txt"))
        for rel in ictal: events[rel]=summary.get(Path(rel).name,[])
    core=[r for r in records if not r.startswith("chb24/")]
    core_seiz=[r for r in seizure_records if not r.startswith("chb24/")]
    provenance={"physionet_version":"1.0.0","records_current":len(records),"seizure_edf_current":len(seizure_records),
                "records_core_without_chb24":len(core),"seizure_edf_core_without_chb24":len(core_seiz),
                "selected_edf":len(set(selected)),"includes_chb24":any(r.startswith("chb24/") for r in selected)}
    return sorted(set(selected)),events,checksums,provenance

def resumable_download(rel,expected,retries=6):
    dest=CACHE/rel; part=dest.with_suffix(dest.suffix+".part"); dest.parent.mkdir(parents=True,exist_ok=True)
    if dest.exists() and expected and sha256_file(dest)==expected: return dest
    if dest.exists():
        if part.exists(): part.unlink()
        dest.replace(part)
    curl=shutil.which("curl.exe") or shutil.which("curl")
    if curl:
        for integrity_attempt in range(2):
            command=[curl,"--fail","--location","--continue-at","-","--retry",str(retries),
                     "--retry-delay","2","--retry-all-errors","--connect-timeout","30","--max-time","1800",
                     "--output",str(part),DATA_URL+rel]
            subprocess.run(command,check=True)
            if not expected or sha256_file(part)==expected:
                os.replace(part,dest); return dest
            # El parcial no puede reutilizarse: se elimina solo este archivo corrupto
            # y se realiza una descarga completa antes de declarar el fallo.
            part.unlink(missing_ok=True)
        raise RuntimeError(f"SHA256 no coincide después de descarga completa: {rel}")
    for attempt in range(retries):
        try:
            offset=part.stat().st_size if part.exists() else 0
            headers={"Range":f"bytes={offset}-"} if offset else {}
            req=urllib.request.Request(DATA_URL+rel,headers=headers)
            with urllib.request.urlopen(req,timeout=180) as response:
                append=offset>0 and getattr(response,"status",200)==206
                with part.open("ab" if append else "wb") as out:
                    shutil.copyfileobj(response,out,length=8*1024*1024)
            if expected and sha256_file(part)!=expected: raise IOError("SHA256 no coincide")
            os.replace(part,dest); return dest
        except Exception as exc:
            if attempt+1==retries: raise RuntimeError(f"No se descargó {rel}: {exc}") from exc
            time.sleep(min(60,2**attempt))

def ensure_downloads(selected,checksums):
    from concurrent.futures import ThreadPoolExecutor,as_completed
    pending=[]
    for rel in selected:
        p=CACHE/rel; expected=checksums.get(rel)
        if not p.exists() or not expected or sha256_file(p)!=expected: pending.append(rel)
    print("EDF seleccionados:",len(selected),"pendientes:",len(pending))
    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
        jobs={pool.submit(resumable_download,rel,checksums.get(rel)):rel for rel in pending}
        for i,fut in enumerate(as_completed(jobs),1):
            fut.result(); print(f"descarga {i}/{len(jobs)}",jobs[fut])
    bad=[rel for rel in selected if not (CACHE/rel).exists() or sha256_file(CACHE/rel)!=checksums.get(rel)]
    if bad: raise RuntimeError(f"EDF sin integridad: {bad[:8]}")

In [ ]:
# BLOQUE 3 — construcción real y perfil exclusivo de desarrollo
def reservoir_add(store,metas,item,meta,seen,limit,rng):
    seen+=1
    if len(store)<limit: store.append(item); metas.append(meta)
    else:
        j=int(rng.integers(0,seen))
        if j<limit: store[j]=item; metas[j]=meta
    return seen

def build_real(scope,out_dir):
    selected,events,checksums,provenance=official_index(scope)
    ensure_downloads(selected,checksums)
    if out_dir.exists(): shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True)
    persons=sorted(set(CASE_PERSON.get(r.split("/")[0],r.split("/")[0]) for r in selected))
    windows=[]; metadata=[]
    for person in persons:
        rng=np.random.default_rng(SEED+stable_int(person)); pos=[]; neg=[]; pm=[]; nm=[]; sp=sn=0
        files=[r for r in selected if CASE_PERSON.get(r.split("/")[0],r.split("/")[0])==person]
        for rel in files:
            raw=mne.io.read_raw_edf(CACHE/rel,preload=True,verbose="ERROR"); raw.pick("eeg")
            data=raw.get_data(); fs_in=float(raw.info["sfreq"])
            if fs_in!=FS: data=mne.filter.resample(data,down=fs_in/FS,npad="auto",axis=1)
            x=common_bipolar(data*1e6,raw.ch_names)
            if x is None:
                print("SKIP montaje incompleto",rel); del raw,data; gc.collect(); continue
            rel_events=events.get(rel,[]); control=len(rel_events)==0
            for s in range(0,len(x)-WIN_PTS+1,WIN_PTS):
                base={"subject_id":person,"case":rel.split("/")[0],"file":rel,"start_s":s/FS}
                if control:
                    sn=reservoir_add(neg,nm,x[s:s+WIN_PTS],{**base,"label":0,"event_id":"","ictal_fraction":0.0},sn,MAX_WINDOWS_PER_SUBJECT_CLASS,rng)
                else:
                    frac=sum(max(0,min((s+WIN_PTS)/FS,b)-max(s/FS,a)) for a,b in rel_events)/WIN_SEC
                    if frac>=0.5:
                        eid=next((j for j,(a,b) in enumerate(rel_events) if max(s/FS,a)<min((s+WIN_PTS)/FS,b)),0)
                        sp=reservoir_add(pos,pm,x[s:s+WIN_PTS],{**base,"label":1,"event_id":f"{rel}:{eid}","ictal_fraction":frac},sp,MAX_WINDOWS_PER_SUBJECT_CLASS,rng)
            del raw,data,x; gc.collect()
        if pos and neg:
            windows.extend(pos+neg); metadata.extend(pm+nm)
        print(person,"ictal",len(pos),"control",len(neg))
    if not windows: raise RuntimeError("No se construyeron ventanas reales")
    X=np.asarray(windows,dtype=np.float32); meta=pd.DataFrame(metadata); y=meta.label.to_numpy(np.int8); pid=meta.subject_id.to_numpy(str)
    np.save(out_dir/"X.npy",X); np.save(out_dir/"y.npy",y); np.save(out_dir/"pid.npy",pid); meta.to_csv(out_dir/"metadata.csv",index=False)
    artifacts={
        "X":{"sha256":sha256_file(out_dir/"X.npy"),"bytes":(out_dir/"X.npy").stat().st_size,"shape":list(X.shape),"dtype":str(X.dtype)},
        "y":{"sha256":sha256_file(out_dir/"y.npy"),"bytes":(out_dir/"y.npy").stat().st_size,"shape":list(y.shape),"dtype":str(y.dtype)},
        "pid":{"sha256":sha256_file(out_dir/"pid.npy"),"bytes":(out_dir/"pid.npy").stat().st_size,"shape":list(pid.shape),"dtype":str(pid.dtype)},
        "metadata":{"sha256":sha256_file(out_dir/"metadata.csv"),"bytes":(out_dir/"metadata.csv").stat().st_size,"rows":len(meta)},
    }
    manifest={"schema_version":"5.1-real","scope":scope,"fs_hz":FS,"window_s":WIN_SEC,"n_windows":len(y),
              "subjects":sorted(set(pid.tolist())),"source":provenance,"selected_files":selected,
              "selected_file_sha256":{r:checksums[r] for r in selected},"artifacts":artifacts,
              "created_utc":time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime())}
    (out_dir/"manifest.json").write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding="utf-8")
    return X,y,pid,meta,manifest

def promote_named(work,mapping):
    backup=WORK_ROOT/"promotion_backup"; backup.mkdir(parents=True,exist_ok=True); moved=[]
    try:
        for src_name,dst in mapping.items():
            src=work/src_name
            if dst.exists():
                b=backup/dst.name; os.replace(dst,b); moved.append((b,dst))
            os.replace(src,dst)
    except Exception:
        for b,dst in reversed(moved):
            if dst.exists(): dst.unlink()
            os.replace(b,dst)
        raise
    shutil.rmtree(backup,ignore_errors=True)

def write_dev_profile(X,y,pid,meta,manifest):
    F=extract_features(X,zscore=False,tag="DEV_raw")
    rows=[]
    for subject in sorted(set(pid.tolist())):
        for cls in (0,1):
            z=F[(pid==subject)&(y==cls)]
            if not len(z): continue
            for j,name in enumerate(FEATURE_NAMES):
                rows.append({"subject_id":subject,"class":cls,"feature":name,"median":float(np.median(z[:,j])),
                             "iqr":float(stats.iqr(z[:,j])),"q05":float(np.percentile(z[:,j],5)),"q95":float(np.percentile(z[:,j],95)),"n":len(z)})
    df=pd.DataFrame(rows); tmp=DEV_WORK/"dev_feature_profile.csv"; df.to_csv(tmp,index=False)
    window_features=meta.reset_index(drop=True).copy()
    for j,name in enumerate(FEATURE_NAMES): window_features[name]=F[:,j]
    window_features.to_csv(DEV_WORK/"dev_window_features.csv",index=False)
    event_rows=[]
    ictal=window_features[(window_features.label==1)&window_features.event_id.notna()&(window_features.event_id.astype(str)!="")]
    for event_id,g in ictal.groupby("event_id"):
        g=g.sort_values("start_s"); q=max(1,len(g)//3); first=g.iloc[:q]; last=g.iloc[-q:]
        event_rows.append({"event_id":event_id,"subject_id":g.subject_id.iloc[0],"n_windows":len(g),
                           "freq_start_hz":float(first.dom_freq.median()),"freq_end_hz":float(last.dom_freq.median()),
                           "ptp_start_uv":float(first.ptp_med.median()),"ptp_end_uv":float(last.ptp_med.median()),
                           "frequency_decreased":bool(first.dom_freq.median()>last.dom_freq.median()),
                           "amplitude_increased":bool(last.ptp_med.median()>first.ptp_med.median())})
    pd.DataFrame(event_rows).to_csv(DEV_WORK/"dev_event_evolution.csv",index=False)
    profile={"schema_version":"5.1-dev","subjects":sorted(set(pid.tolist())),"n_windows":int(len(y)),
             "source_manifest_sha256":sha256_file(DEV_WORK/"manifest.json"),"feature_names":FEATURE_NAMES,
             "feature_domains":FEATURE_DOMAINS,"n_events_with_evolution":len(event_rows),
             "created_utc":time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime())}
    (DEV_WORK/"development_profile.json").write_text(json.dumps(profile,indent=2,ensure_ascii=False),encoding="utf-8")
    promote_named(DEV_WORK,{"dev_feature_profile.csv":OUT_DIR/"dev_feature_profile.csv",
                            "dev_window_features.csv":OUT_DIR/"dev_window_features.csv",
                            "dev_event_evolution.csv":OUT_DIR/"dev_event_evolution.csv",
                            "development_profile.json":OUT_DIR/"development_profile.json",
                            "manifest.json":OUT_DIR/"dev_source_manifest.json",
                            "metadata.csv":OUT_DIR/"dev_window_metadata.csv"})
    shutil.rmtree(DEV_WORK,ignore_errors=True)
    print("DESARROLLO: SUPPORTED",profile)

REAL={"ready":False,"reason":"modo sin cohorte externa"}
if MODE=="prepare_dev":
    Xd,yd,pd_,md,mf=build_real("dev",DEV_WORK); write_dev_profile(Xd,yd,pd_,md,mf)
elif MODE=="external":
    paths=[REAL_DIR/"X_real.npy",REAL_DIR/"y_real.npy",REAL_DIR/"pid_real.npy",REAL_DIR/"real_window_metadata.csv",REAL_DIR/"real_manifest.json"]
    if REBUILD_REAL or not all(p.exists() for p in paths):
        Xr,yr,pr,mr,mf=build_real("all",REAL_WORK)
        if len(set(pr.tolist()))<15: raise RuntimeError("cohorte externa insuficiente")
        promote_named(REAL_WORK,{"X.npy":paths[0],"y.npy":paths[1],"pid.npy":paths[2],"metadata.csv":paths[3],"manifest.json":paths[4]})
        shutil.rmtree(REAL_WORK,ignore_errors=True)
    Xr=np.load(paths[0],mmap_mode="r"); yr=np.load(paths[1]); pr=np.load(paths[2]); mr=pd.read_csv(paths[3]); rm=json.loads(paths[4].read_text())
    role_paths={"X":paths[0],"y":paths[1],"pid":paths[2],"metadata":paths[3]}
    if set(rm.get("artifacts",{}))!=set(role_paths): raise RuntimeError("manifiesto REAL sin hashes completos")
    bad=[role for role,p in role_paths.items() if sha256_file(p)!=rm["artifacts"][role].get("sha256")]
    if bad: raise RuntimeError(f"artefactos REAL con SHA256 inválido: {bad}")
    if Xr.shape!=(len(yr),int(FS*WIN_SEC),len(BIPOLAR)) or len(pr)!=len(yr) or len(mr)!=len(yr):
        raise RuntimeError("formas REAL incompatibles")
    if set(np.unique(yr).tolist())!={0,1}: raise RuntimeError("clases REAL incompletas")
    for start in range(0,len(Xr),1000):
        if not np.isfinite(np.asarray(Xr[start:start+1000])).all(): raise RuntimeError("NaN/Inf en REAL")
    if set(pr.tolist())!=set(mr.subject_id.astype(str)): raise RuntimeError("IDs REAL incompatibles")
    REAL={"ready":True,"X":Xr,"y":yr,"pid":pr,"meta":mr,"manifest":rm,"subjects":sorted(set(pr.tolist())),
          "manifest_sha256":sha256_file(paths[4])}
print("REAL:","SUPPORTED" if REAL["ready"] else "NOT EVALUABLE — "+REAL["reason"])

In [ ]:
# BLOQUE 4 — fidelidad, cobertura morfológica, diversidad y proximidad
def sample_idx(y,n,seed):
    rng=np.random.default_rng(seed); out=[]
    for cls in (0,1):
        a=np.flatnonzero(np.asarray(y)==cls); out.extend(rng.choice(a,min(n,len(a)),replace=False).tolist())
    rng.shuffle(out); return np.asarray(out,dtype=int)

def synthetic_samples(n=1200):
    Xs=[]; ys=[]; scenarios=[]; split_tags=[]
    for k in ("train","val","test"):
        idx=sample_idx(SYN["y"][k],n,SEED+len(Xs))
        Xs.append(synthetic_to_bipolar(np.asarray(SYN["X"][k][idx])))
        ys.append(SYN["y"][k][idx])
        scenarios.extend(SYN["meta"][k].iloc[idx].scenario.astype(str).tolist())
        split_tags.extend([k]*len(idx))
    X=np.concatenate(Xs); y=np.concatenate(ys)
    return X,y,np.asarray(scenarios),np.asarray(split_tags)

def internal_separability(Fs_raw,Fs_z,ys,split_tags):
    # Evaluación exclusivamente entre pacientes sintéticos disjuntos: TRAIN+VAL -> TEST.
    tr=np.flatnonzero(split_tags!="test"); te=np.flatnonzero(split_tags=="test"); rows=[]
    for mode,F in (("raw",Fs_raw),("zscore",Fs_z)):
        for model in ("LR","RF"):
            auc,ap=clf_scores(F[tr],ys[tr],F[te],ys[te],model,SEED)
            rows.append({"mode":mode,"model":model,"protocol":"SYN_TRAIN+VAL→SYN_TEST",
                         "roc_auc":auc,"average_precision":ap,"n_train":len(tr),"n_test":len(te),
                         "train_splits":"train,val","test_split":"test","patient_disjoint":True,
                         "status":"SUPPORTED" if np.isfinite(auc) else "NOT EVALUABLE"})
    return pd.DataFrame(rows)

def cliffs_delta(a,b):
    a=np.asarray(a); b=np.asarray(b); d=a[:,None]-b[None,:]
    return float((np.sum(d>0)-np.sum(d<0))/d.size)

def mmd2_rbf(X,Y):
    Z=np.vstack([X,Y]); D=cdist(Z,Z); pos=D[D>0]; gamma=1/(2*np.median(pos)**2+1e-12)
    Kx=np.exp(-gamma*cdist(X,X,"sqeuclidean")); Ky=np.exp(-gamma*cdist(Y,Y,"sqeuclidean")); Kxy=np.exp(-gamma*cdist(X,Y,"sqeuclidean")); np.fill_diagonal(Kx,0); np.fill_diagonal(Ky,0)
    return float(Kx.sum()/(len(X)*(len(X)-1))+Ky.sum()/(len(Y)*(len(Y)-1))-2*Kxy.mean())

def distribution_metrics(Fs,Fr,ys,yr,scenarios,pid):
    rows=[]; coverage=[]
    groups=[("interictal",ys==0,yr==0),("ictal_all",ys==1,yr==1),
            ("generalized_absence",(ys==1)&(scenarios=="generalized_absence"),yr==1),
            ("focal_temporal",(ys==1)&(scenarios=="focal_temporal"),yr==1)]
    rng=np.random.default_rng(SEED)
    for group,sm,rm in groups:
        si=np.flatnonzero(sm); ri=np.flatnonzero(rm); si=rng.choice(si,min(600,len(si)),False); ri=rng.choice(ri,min(600,len(ri)),False)
        for j,name in enumerate(FEATURE_NAMES):
            a=Fs[si,j]; b=Fr[ri,j]; q05s,q95s=np.percentile(a,[5,95]); q05r,q95r=np.percentile(b,[5,95])
            rows.append({"group":group,"feature":name,"wasserstein":float(stats.wasserstein_distance(a,b)),
                         "wasserstein_over_real_iqr":float(stats.wasserstein_distance(a,b)/(stats.iqr(b)+1e-12)),
                         "ks_stat":float(stats.ks_2samp(a,b).statistic),"cliffs_delta":cliffs_delta(a,b)})
            coverage.append({"group":group,"feature":name,"real_inside_syn_05_95":float(np.mean((b>=q05s)&(b<=q95s))),
                             "syn_inside_real_05_95":float(np.mean((a>=q05r)&(a<=q95r)))})
    subject_rows=[]
    for subject in sorted(set(pid.tolist())):
        for cls in (0,1):
            rr=np.flatnonzero((yr==cls)&(pid==subject)); ss=np.flatnonzero(ys==cls)
            if not len(rr) or not len(ss): continue
            for j,name in enumerate(FEATURE_NAMES):
                subject_rows.append({"subject":subject,"class":cls,"feature":name,
                    "wasserstein_over_real_iqr":float(stats.wasserstein_distance(Fs[ss,j],Fr[rr,j])/(stats.iqr(Fr[rr,j])+1e-12)),
                    "ks_stat":float(stats.ks_2samp(Fs[ss,j],Fr[rr,j]).statistic),"cliffs_delta":cliffs_delta(Fs[ss[:min(400,len(ss))],j],Fr[rr[:min(400,len(rr))],j])})
    return pd.DataFrame(rows),pd.DataFrame(coverage),pd.DataFrame(subject_rows)

def proximity_diversity(Fs,Fr,ys,yr):
    rng=np.random.default_rng(SEED); rows=[]; scaler=StandardScaler().fit(Fr)
    for cls in (0,1):
        si=rng.choice(np.flatnonzero(ys==cls),min(500,np.sum(ys==cls)),False); ri=rng.choice(np.flatnonzero(yr==cls),min(500,np.sum(yr==cls)),False)
        xs=scaler.transform(Fs[si]); xr=scaler.transform(Fr[ri]); rr=NearestNeighbors(n_neighbors=2).fit(xr).kneighbors(xr)[0][:,1]; sr=NearestNeighbors(n_neighbors=1).fit(xr).kneighbors(xs)[0][:,0]; ss=NearestNeighbors(n_neighbors=2).fit(xs).kneighbors(xs)[0][:,1]
        threshold=float(np.percentile(rr,5)); rows.append({"class":cls,"fraction_close_feature_space":float(np.mean(sr<threshold)),
            "syn_real_nn_median":float(np.median(sr)),"real_real_nn_median":float(np.median(rr)),"syn_syn_nn_median":float(np.median(ss)),
            "diversity_ratio_syn_real":float(np.median(ss)/(np.median(rr)+1e-12)),"mmd2_rbf":mmd2_rbf(xs,xr),"not_privacy":True})
    return pd.DataFrame(rows)

In [ ]:
# BLOQUE 5 — protocolos LOSO y bootstrap por persona
def balanced(y,n,seed):
    rng=np.random.default_rng(seed); out=[]
    for cls in (0,1):
        a=np.flatnonzero(np.asarray(y)==cls); out.extend(rng.choice(a,min(n,len(a)),False).tolist())
    rng.shuffle(out); return np.asarray(out,dtype=int)

def smote_training_only(X,y,seed):
    from imblearn.over_sampling import SMOTE
    counts=np.bincount(np.asarray(y,dtype=int),minlength=2)
    if counts.min()<2 or counts[0]==counts[1]: return np.asarray(X),np.asarray(y)
    return SMOTE(random_state=seed,k_neighbors=min(5,int(counts.min())-1)).fit_resample(np.asarray(X),np.asarray(y))

def clf_scores(Xtr,ytr,Xte,yte,name,seed):
    if len(np.unique(ytr))<2 or len(np.unique(yte))<2: return np.nan,np.nan
    model=(make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000,class_weight="balanced",random_state=seed)) if name=="LR" else RandomForestClassifier(n_estimators=300,n_jobs=-1,class_weight="balanced",random_state=seed))
    model.fit(Xtr,ytr); p=model.predict_proba(Xte)[:,1]
    return float(roc_auc_score(yte,p)),float(average_precision_score(yte,p))

def bootstrap_subject(values,seed=SEED,n_boot=2000):
    a=np.asarray(values,float); a=a[np.isfinite(a)]
    if not len(a): return np.nan,np.nan,np.nan
    rng=np.random.default_rng(seed); boot=np.asarray([a[rng.integers(0,len(a),len(a))].mean() for _ in range(n_boot)])
    return float(a.mean()),float(np.percentile(boot,2.5)),float(np.percentile(boot,97.5))

def run_protocols(Fs_raw,Fs_z,ys,Fr_raw,Fr_z,yr,pid):
    dev=set(DEV_CASES); external=sorted(set(pid.tolist())-dev); rows=[]
    if set(pid.tolist()) & dev: raise RuntimeError("los sujetos de desarrollo llegaron a run_protocols")
    for subject in external:
        test=np.flatnonzero(pid==subject); train=np.flatnonzero(pid!=subject); seed=SEED+stable_int(subject)%10000
        # R2R conserva la prevalencia observada. El límite es computacional y se muestrea
        # sin usar el sujeto de prueba. SMOTE se ajusta después y solo sobre este training.
        rng=np.random.default_rng(seed)
        train=rng.choice(train,min(4000,len(train)),replace=False)
        if len(np.unique(yr[train]))<2: raise RuntimeError(f"training LOSO monoclase: {subject}")
        syn_take=balanced(ys,3000,seed+1)
        for mode,Fs,Fr in (("raw",Fs_raw,Fr_raw),("zscore",Fs_z,Fr_z)):
            Xrtr,yrtr=Fr[train],yr[train]; Xrte,yrte=Fr[test],yr[test]; Xst,yst=Fs[syn_take],ys[syn_take]
            Xsm,ysm=smote_training_only(Xrtr,yrtr,seed)
            specs={"R2R":(Xrtr,yrtr,Xrte,yrte),"TSTR":(Xst,yst,Xrte,yrte),"TRTS":(Xrtr,yrtr,Fs,ys),
                   "R+S→R":(np.vstack([Xrtr,Xst]),np.concatenate([yrtr,yst]),Xrte,yrte),"SMOTE→R":(Xsm,ysm,Xrte,yrte)}
            for model in ("LR","RF"):
                for protocol,(a,b,c,d) in specs.items():
                    auc,ap=clf_scores(a,b,c,d,model,seed); rows.append({"subject":subject,"mode":mode,"model":model,"protocol":protocol,"roc_auc":auc,"average_precision":ap,"n_test":len(d)})
    detail=pd.DataFrame(rows); summaries=[]
    for keys,g in detail.groupby(["mode","model","protocol"]):
        for metric in ("roc_auc","average_precision"):
            mean,lo,hi=bootstrap_subject(g[metric]); summaries.append({"mode":keys[0],"model":keys[1],"protocol":keys[2],"metric":metric,"macro_mean":mean,"ci95_low":lo,"ci95_high":hi,"n_subjects":g.subject.nunique()})
    return detail,pd.DataFrame(summaries)

In [ ]:
# BLOQUE 6 — ejecución externa, estados y auditoría de afirmaciones
summary={"mode":MODE,"synthetic_status":"SUPPORTED" if SYN["ready"] else "NOT EVALUABLE",
         "real_status":"SUPPORTED" if REAL["ready"] else "NOT EVALUABLE","external_validation_status":"NOT EVALUABLE",
         "claims":{},"generated_utc":time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime())}

if MODE=="external" and SYN["ready"] and REAL["ready"]:
    Xs,ys,scenarios,split_tags=synthetic_samples()
    pid_all=np.asarray(REAL["pid"]); external_mask=~np.isin(pid_all,np.asarray(sorted(DEV_CASES)))
    Xr=np.asarray(REAL["X"])[external_mask]; yr=np.asarray(REAL["y"])[external_mask]; pid=pid_all[external_mask]
    if set(pid.tolist()) & set(DEV_CASES): raise RuntimeError("fuga de sujetos de desarrollo a evaluación externa")
    if len(set(pid.tolist()))!=18: raise RuntimeError(f"se esperaban 18 sujetos externos y hay {len(set(pid.tolist()))}")
    Fs_raw=extract_features(Xs,False,"SYN_raw"); Fs_z=extract_features(Xs,True,"SYN_zscore")
    Fr_raw=extract_features(Xr,False,"REAL_raw"); Fr_z=extract_features(Xr,True,"REAL_zscore")
    internal_df=internal_separability(Fs_raw,Fs_z,ys,split_tags)
    fidelity_df,coverage_df,subject_fidelity_df=distribution_metrics(Fs_raw,Fr_raw,ys,yr,scenarios,pid)
    proximity_df=proximity_diversity(Fs_raw,Fr_raw,ys,yr)
    protocol_df,protocol_summary_df=run_protocols(Fs_raw,Fs_z,ys,Fr_raw,Fr_z,yr,pid)
    internal_df.to_csv(OUT_DIR/"01_internal_separability.csv",index=False)
    fidelity_df.to_csv(OUT_DIR/"02_fidelity_featurewise.csv",index=False); coverage_df.to_csv(OUT_DIR/"02b_morphology_coverage.csv",index=False)
    subject_fidelity_df.to_csv(OUT_DIR/"02c_fidelity_by_subject.csv",index=False); proximity_df.to_csv(OUT_DIR/"04_feature_space_proximity_diversity.csv",index=False)
    protocol_df.to_csv(OUT_DIR/"03_cross_domain_loso.csv",index=False); protocol_summary_df.to_csv(OUT_DIR/"03b_cross_domain_bootstrap.csv",index=False)
    target=protocol_summary_df[(protocol_summary_df["mode"]=="zscore")&(protocol_summary_df.model=="RF")&(protocol_summary_df.protocol=="TSTR")&(protocol_summary_df.metric=="roc_auc")].iloc[0]
    tstr_status="SUPPORTED" if target.macro_mean>0.5 and target.ci95_low>0.5 else "NOT SUPPORTED"
    ictal_fid=fidelity_df[fidelity_df.group=="ictal_all"]
    large_ictal_effects=int((ictal_fid.cliffs_delta.abs()>=0.474).sum())
    high_ictal_ks=int((ictal_fid.ks_stat>=0.5).sum())
    fidelity_status="NOT SUPPORTED" if (large_ictal_effects or high_ictal_ks) else "SUPPORTED"
    low_coverage=int(((coverage_df.real_inside_syn_05_95<0.5)|(coverage_df.syn_inside_real_05_95<0.5)).sum())
    coverage_status="NOT SUPPORTED" if low_coverage else "SUPPORTED"
    low_diversity=int((proximity_df.diversity_ratio_syn_real<0.8).sum())
    diversity_status="NOT SUPPORTED" if low_diversity else "SUPPORTED"
    internal_primary=internal_df[(internal_df["mode"]=="zscore")&(internal_df.model=="RF")].iloc[0]
    summary["external_validation_status"]="SUPPORTED"
    summary["claims"]={
        "generator_integrity":{"status":"SUPPORTED","manifest_sha256":sha256_file(SYN_DIR/"generation_manifest.json")},
        "real_cohort_integrity":{"status":"SUPPORTED","manifest_sha256":REAL["manifest_sha256"],"n_windows_all":int(len(REAL["y"])),"n_windows_external":int(len(yr)),"n_external_subjects":int(len(set(pid.tolist()))),"development_subjects_excluded":True},
        "internal_separability":{"status":"SUPPORTED","primary_auc":float(internal_primary.roc_auc),"file":"01_internal_separability.csv","not_external_validity":True},
        "external_TSTR_RF_zscore":{"status":tstr_status,"macro_auc":float(target.macro_mean),"ci95":[float(target.ci95_low),float(target.ci95_high)],"n_subjects":int(target.n_subjects)},
        "fidelity_similarity":{"status":fidelity_status,"large_ictal_cliff_effects":large_ictal_effects,"ictal_features_ks_ge_0_5":high_ictal_ks,"file":"02_fidelity_featurewise.csv"},
        "external_morphology_coverage":{"status":coverage_status,"feature_pairs_with_any_coverage_below_0_5":low_coverage,"interpretation":"cobertura cuantitativa; no diagnóstico clínico","file":"02b_morphology_coverage.csv"},
        "feature_space_proximity":{"status":"SUPPORTED","not_privacy":True,"file":"04_feature_space_proximity_diversity.csv"},
        "diversity_vs_real":{"status":diversity_status,"classes_below_0_8":low_diversity,"file":"04_feature_space_proximity_diversity.csv"},
    }
    favorable_all=(tstr_status=="SUPPORTED" and fidelity_status=="SUPPORTED" and coverage_status=="SUPPORTED" and diversity_status=="SUPPORTED")
    framework_verdict="VIABLE" if favorable_all else "VIABLE CON LIMITACIONES"
    framework_reason=("El framework operacional es reproducible y transferencia, fidelidad, cobertura y diversidad satisfacen los criterios declarados." if favorable_all else
                      "El generador es reproducible y TSTR está respaldado, pero fidelidad, cobertura externa y diversidad muestran desplazamiento de dominio.")
    summary["framework_verdict"]={"status":framework_verdict,"reason":framework_reason,"not_clinical_equivalence":True}
    # La tesis no se modifica en esta fase y aún contiene cifras históricas incompatibles.
    summary["doctoral_verdict"]={"status":"NO VIABLE EN SU REDACCIÓN ACTUAL",
        "reason":"El manuscrito proporcionado conserva resultados históricos obsoletos; debe sustituirlos por la matriz 06 antes de defender las conclusiones.",
        "conditional_after_revision":framework_verdict,"not_clinical_equivalence":True}
    smote_primary=protocol_summary_df[(protocol_summary_df["mode"]=="zscore")&(protocol_summary_df.model=="RF")&(protocol_summary_df.protocol=="SMOTE→R")&(protocol_summary_df.metric=="roc_auc")].iloc[0]
    r2r_primary=protocol_summary_df[(protocol_summary_df["mode"]=="zscore")&(protocol_summary_df.model=="RF")&(protocol_summary_df.protocol=="R2R")&(protocol_summary_df.metric=="roc_auc")].iloc[0]
    prox0=proximity_df[proximity_df["class"]==0].iloc[0]; prox1=proximity_df[proximity_df["class"]==1].iloc[0]
    claims=pd.DataFrame([
        {"thesis_claim":"N=3000 sujetos sintéticos","status":"VIGENTE","new_evidence":"generation_manifest.json","required_action":"mantener"},
        {"thesis_claim":"CHB-MIT limitado a 5 sujetos/24 EDF","status":"OBSOLETO","new_evidence":"23 personas procesadas; 18 externas; 171 EDF seleccionados","required_action":"sustituir métodos y limitaciones"},
        {"thesis_claim":"ventanas REAL con 50% de solapamiento","status":"OBSOLETO","new_evidence":"2 s sin solapamiento","required_action":"sustituir"},
        {"thesis_claim":"ROC-AUC interno histórico 0.999","status":"OBSOLETO","new_evidence":f"RF zscore={internal_primary.roc_auc:.6f}","required_action":"sustituir por 01_internal_separability.csv"},
        {"thesis_claim":"TSTR histórico 0.613","status":"OBSOLETO","new_evidence":f"RF zscore={target.macro_mean:.6f}; IC95% [{target.ci95_low:.6f}, {target.ci95_high:.6f}]","required_action":"sustituir"},
        {"thesis_claim":"R2R histórico 0.466","status":"OBSOLETO","new_evidence":f"RF zscore={r2r_primary.macro_mean:.6f}; IC95% [{r2r_primary.ci95_low:.6f}, {r2r_primary.ci95_high:.6f}]","required_action":"sustituir"},
        {"thesis_claim":"SMOTE histórico 0.475 ± 0.098","status":"OBSOLETO","new_evidence":f"RF zscore={smote_primary.macro_mean:.6f}; IC95% [{smote_primary.ci95_low:.6f}, {smote_primary.ci95_high:.6f}]","required_action":"sustituir"},
        {"thesis_claim":"proximidad/memorización histórica 0.8%","status":"OBSOLETO","new_evidence":f"proximidad clase0={prox0.fraction_close_feature_space:.4f}; clase1={prox1.fraction_close_feature_space:.4f}","required_action":"renombrar; no afirmar privacidad"},
        {"thesis_claim":"diversidad histórica 0.569","status":"OBSOLETO","new_evidence":f"SYN/REAL clase0={prox0.diversity_ratio_syn_real:.4f}; clase1={prox1.diversity_ratio_syn_real:.4f}","required_action":"sustituir y declarar diversidad menor"},
        {"thesis_claim":"Gap/IQR global histórico 1.045 y rasgos históricos","status":"OBSOLETO","new_evidence":"02_fidelity_featurewise.csv","required_action":"sustituir cifras completas"},
        {"thesis_claim":"equivalencia clínica o capacidad diagnóstica","status":"NO RESPALDADO","new_evidence":"fuera del alcance","required_action":"no afirmar"},
        {"thesis_claim":"fidelidad uniforme SYN↔REAL","status":"NO RESPALDADO","new_evidence":f"{large_ictal_effects} efectos ictales Cliff grandes; {high_ictal_ks} KS≥0.5","required_action":"reportar desplazamiento de dominio"},
        {"thesis_claim":"generador paramétrico controlado para investigación","status":"VIGENTE","new_evidence":"integridad, reproducibilidad y contrato interno aprobados","required_action":"mantener con limitaciones externas"},
    ])
    claims.to_csv(OUT_DIR/"06_thesis_claim_matrix.csv",index=False)
elif not SYN["ready"]:
    summary["claims"]["generator_integrity"]={"status":"NOT EVALUABLE","reason":SYN["reason"]}
elif MODE=="prepare_dev":
    summary["claims"]["development_profile"]={"status":"SUPPORTED","file":"development_profile.json"}
else:
    summary["claims"]["external_validation"]={"status":"NOT EVALUABLE","reason":"ejecutar modo external después de congelar el generador"}

(OUT_DIR/"05_final_summary.json").write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding="utf-8")
pd.DataFrame([{"claim":k,"status":v.get("status")} for k,v in summary["claims"].items()]).to_csv(OUT_DIR/"05_final_verdict_lines.csv",index=False)
print(json.dumps(summary,indent=2,ensure_ascii=False))